# Notebook 01 — Data Understanding
## Telecom Customer Churn and Retention Intelligence Dashboard

---

**Purpose:** Understand the raw dataset structure, inspect data types, identify quality issues, and document findings before any cleaning or transformation.

**Input:** `data/raw/Telco-Customer-Churn.csv`

**Output:** Documented findings fed into `reports/data_quality_report.md`

**Rule:** No cleaning happens in this notebook. We only observe and document.

---

## Section 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries imported successfully.')

## Section 2 — Load Raw Dataset

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'Telco-Customer-Churn.csv'

# Always work on a copy — never mutate df_raw
df_raw = pd.read_csv(RAW_FILE)
df = df_raw.copy()

print(f'File loaded: {RAW_FILE}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Section 3 — Preview Rows

Always look at the first few rows, last few rows, and a random sample to get a feel for the data.

In [ ]:
print('--- First 5 rows ---')
display(df.head())

In [ ]:
print('--- Last 5 rows ---')
display(df.tail())

In [ ]:
print('--- Random 5 rows (reproducible sample) ---')
display(df.sample(5, random_state=42))

## Section 4 — Dataset Shape and Column List

In [ ]:
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print()
print('Column Names:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:02d}. {col}')

## Section 5 — Data Types

Understanding data types is critical. A column stored as `object` (text) when it should be `float64` (number) will break all numeric calculations.

In [ ]:
df.info()

In [ ]:
# Summary of dtype categories
dtype_summary = df.dtypes.value_counts().reset_index()
dtype_summary.columns = ['Data Type', 'Count']
display(dtype_summary)

## Section 6 — Descriptive Statistics

Statistics like mean, min, max help spot obvious issues (e.g. negative tenure, impossibly high charges).

In [ ]:
print('--- Numeric Columns ---')
display(df.describe(include=[np.number]))

In [ ]:
print('--- Text / Categorical Columns ---')
display(df.describe(include='object'))

## Section 7 — Missing Value Analysis

Two kinds of missing data exist:
- `NaN` — Python/Pandas null (true missing)
- Blank string `" "` — an empty string that looks like data but isn't

We check both.

In [ ]:
# Standard NaN check
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percent': df.isnull().mean().mul(100).round(2)
})
print('--- NaN / Null Values ---')
display(missing)

In [ ]:
# Blank string check on object columns
print('--- Blank String Check (object columns) ---')
for col in df.select_dtypes(include='object').columns:
    blank_count = df[col].astype(str).str.strip().eq('').sum()
    if blank_count > 0:
        print(f'  {col}: {blank_count} blank rows')
print('Done.')

## Section 8 — Duplicate Analysis

In [ ]:
# Exact row duplicates
exact_dups = df.duplicated().sum()
print(f'Exact duplicate rows     : {exact_dups}')

# Customer ID duplicates
id_dups = df['customerID'].duplicated().sum()
print(f'Duplicate customer IDs   : {id_dups}')

# Missing customer IDs
id_missing = df['customerID'].isna().sum()
print(f'Missing customer IDs     : {id_missing}')

# Unique customer IDs
id_unique = df['customerID'].nunique()
print(f'Unique customer IDs      : {id_unique}')

## Section 9 — Unique Value Analysis

Helps understand cardinality — how many distinct values each column has.

In [ ]:
unique_counts = df.nunique().reset_index()
unique_counts.columns = ['Column', 'Unique Values']
display(unique_counts)

## Section 10 — Category Audit

For every text column (except customer ID), print all unique values and their counts.
This catches typos, inconsistent casing, or unexpected categories.

In [ ]:
skip_cols = ['customerID', 'TotalCharges']  # Non-categorical or already handled

for col in df.select_dtypes(include='object').columns:
    if col in skip_cols:
        continue
    print(f'\n{"="*50}')
    print(f'COLUMN: {col}  (unique values: {df[col].nunique()})')
    print(df[col].value_counts(dropna=False).to_string())

## Section 11 — TotalCharges Deep Inspection

`TotalCharges` is stored as an `object` column. This is a known issue.
We investigate the blank rows and their relationship with `tenure`.

In [ ]:
# Identify blank TotalCharges rows
blank_mask = df['TotalCharges'].astype(str).str.strip().eq('')

print(f'Blank TotalCharges count: {blank_mask.sum()}')
print()
print('Rows with blank TotalCharges:')
display(df.loc[blank_mask, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

In [ ]:
# Confirm all blank rows have tenure = 0
blank_with_nonzero_tenure = df.loc[blank_mask & df['tenure'].ne(0)]
print(f'Blank TotalCharges rows with tenure != 0: {len(blank_with_nonzero_tenure)}')
print('Conclusion: All blank TotalCharges rows have tenure = 0 (new customers not yet billed)')

## Section 12 — Numeric Range Audit

In [ ]:
# TotalCharges as numeric for range check
df_temp = df.copy()
df_temp['TotalCharges'] = pd.to_numeric(df_temp['TotalCharges'], errors='coerce')

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

range_report = []
for col in numeric_cols:
    range_report.append({
        'Column': col,
        'Min': df_temp[col].min(),
        'Max': df_temp[col].max(),
        'Mean': round(df_temp[col].mean(), 2),
        'Median': df_temp[col].median(),
        'Nulls': df_temp[col].isna().sum(),
        'Negatives': df_temp[col].lt(0).sum()
    })

display(pd.DataFrame(range_report))

## Section 13 — Churn Distribution

This is the most important column in the dataset. We analyze its distribution in detail.

In [ ]:
churn_counts = df['Churn'].value_counts(dropna=False)
churn_pct = df['Churn'].value_counts(normalize=True, dropna=False).mul(100).round(2)

churn_summary = pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct})
display(churn_summary)

print(f'\nOverall Churn Rate: {churn_pct["Yes"]:.2f}%')
print(f'Overall Retention Rate: {churn_pct["No"]:.2f}%')

In [ ]:
# Visualize churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart — counts
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Customer Count by Churn Status', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Customer Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, max(churn_counts.values) * 1.15)
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

# Pie chart — percentages
axes[1].pie(
    churn_counts.values,
    labels=[f'Retained\n{churn_pct["No"]:.1f}%', f'Churned\n{churn_pct["Yes"]:.1f}%'],
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
axes[1].set_title('Churn vs Retention Split', fontsize=13, fontweight='bold', pad=12)

plt.suptitle('Overall Churn Distribution — IBM Telco Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'assets' / 'churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to assets/churn_distribution.png')

## Section 14 — Contract, Tenure and Charges Overview

A quick look at the three most important analytical dimensions before deep EDA.

In [ ]:
# Contract distribution
print('Contract Type Distribution:')
display(df['Contract'].value_counts(dropna=False).reset_index().rename(columns={'count': 'Customers'}))

print('\nInternet Service Distribution:')
display(df['InternetService'].value_counts(dropna=False).reset_index().rename(columns={'count': 'Customers'}))

print('\nPayment Method Distribution:')
display(df['PaymentMethod'].value_counts(dropna=False).reset_index().rename(columns={'count': 'Customers'}))

In [ ]:
# Numeric overview chart
df_temp = df.copy()
df_temp['TotalCharges'] = pd.to_numeric(df_temp['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(df_temp['tenure'], bins=30, color='#3498db', edgecolor='white', linewidth=0.5)
axes[0].set_title('Tenure Distribution (Months)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tenure (Months)')
axes[0].set_ylabel('Customer Count')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].hist(df_temp['MonthlyCharges'], bins=30, color='#9b59b6', edgecolor='white', linewidth=0.5)
axes[1].set_title('Monthly Charges Distribution ($)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Customer Count')
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

axes[2].hist(df_temp['TotalCharges'].dropna(), bins=30, color='#e67e22', edgecolor='white', linewidth=0.5)
axes[2].set_title('Total Charges Distribution ($)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Total Charges ($)')
axes[2].set_ylabel('Customer Count')
axes[2].grid(axis='y', alpha=0.3)
axes[2].spines[['top', 'right']].set_visible(False)

plt.suptitle('Numeric Column Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'assets' / 'numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to assets/numeric_distributions.png')

## Section 15 — Data Quality Report Summary

Consolidates all findings from this notebook into a final summary table.

In [ ]:
df_temp2 = df.copy()
df_temp2['TotalCharges'] = pd.to_numeric(df_temp2['TotalCharges'], errors='coerce')

blank_tc = df['TotalCharges'].astype(str).str.strip().eq('').sum()
churn_rate = df['Churn'].eq('Yes').mean() * 100

summary = {
    'Total Rows'             : df.shape[0],
    'Total Columns'          : df.shape[1],
    'Duplicate Rows'         : df.duplicated().sum(),
    'Duplicate Customer IDs' : df['customerID'].duplicated().sum(),
    'Missing Customer IDs'   : df['customerID'].isna().sum(),
    'NaN Values (all cols)'  : df.isnull().sum().sum(),
    'Blank TotalCharges'     : blank_tc,
    'Churn Rate (%)'         : round(churn_rate, 2),
    'Tenure Min'             : df['tenure'].min(),
    'Tenure Max'             : df['tenure'].max(),
    'MonthlyCharges Min ($)' : df['MonthlyCharges'].min(),
    'MonthlyCharges Max ($)' : df['MonthlyCharges'].max(),
    'TotalCharges Min ($)'   : df_temp2['TotalCharges'].min(),
    'TotalCharges Max ($)'   : df_temp2['TotalCharges'].max(),
}

summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])
display(summary_df)

print('\n--- INITIAL FINDINGS ---')
print(f'Dataset is clean with ONE issue: {blank_tc} blank TotalCharges rows (all have tenure=0).')
print('Planned fix: convert to numeric, set blank rows to 0 in the cleaning phase.')

---

## Next Step

Proceed to **`02_data_cleaning.ipynb`** where we:
1. Rename columns to snake_case
2. Trim all string values
3. Remove duplicates
4. Fix the `TotalCharges` data type issue
5. Validate all numeric fields
6. Save the staging output

---
**Notebook complete. No data was modified.**